# MUTCD Multimodal RAG — TAMU HPRC edition

Adapted from the Colab notebook `Copy_of_MRAG.ipynb`. Same dual-retrieval pipeline:
section-level text retrieval + page-level visual retrieval, fused before a VLM (Qwen2.5-VL-3B) produces the final grounded answer.

**Before running** make sure you went through `HPRC_SETUP.md` once:
1. Data in `$SCRATCH/MRAG/` (PDF, sections JSON, optional `page_images/`).
2. Conda env `mrag` created in `$SCRATCH/envs/mrag` from `requirements.txt`.
3. JupyterLab launched via OnDemand with **modules: `Anaconda3 WebProxy`**, **GPU: 1**, and the `mrag` env selected.
4. Kernel for this notebook switched to **Python (mrag)**.

If you skipped the optional SLURM ingestion (`scripts/ingest.slurm`), the notebook will build the page images, page records, and embeddings on its own and cache them under `$SCRATCH/MRAG/mmrag_cache/`.

## 0. Environment — paths, caches, GPU sanity check

Pins all caches into `$SCRATCH` so the small `$HOME` quota is never touched, and prints the GPU we ended up with.

In [ ]:
import os, sys, platform
from pathlib import Path

SCRATCH = Path(os.environ.get("SCRATCH", "/tmp"))
BASE_DIR = SCRATCH / "MRAG"

# Send every cache into scratch BEFORE importing torch / transformers.
os.environ.setdefault("HF_HOME",            str(SCRATCH / "hf_cache"))
os.environ.setdefault("TRANSFORMERS_CACHE", str(SCRATCH / "hf_cache"))
os.environ.setdefault("HF_HUB_CACHE",       str(SCRATCH / "hf_cache" / "hub"))
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
for k in ("HF_HOME", "TRANSFORMERS_CACHE", "HF_HUB_CACHE"):
    Path(os.environ[k]).mkdir(parents=True, exist_ok=True)

import torch
print("python      :", sys.version.split()[0])
print("platform    :", platform.node())
print("torch       :", torch.__version__)
print("cuda avail  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device :", torch.cuda.get_device_name(0))
    print("cuda mem GB :", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1))
print("SCRATCH     :", SCRATCH)
print("BASE_DIR    :", BASE_DIR, "exists?", BASE_DIR.exists())
print("HF_HOME     :", os.environ['HF_HOME'])

## 1. Configuration

Same knobs as the Colab notebook, just pointed at `$SCRATCH`. Edit any of these if you renamed files or want different retrieval depth.

In [ ]:
# ----- Paths -----
PDF_PATH        = BASE_DIR / "mutcd11theditionr1hl.pdf"
SECTIONS_JSON   = BASE_DIR / "mutcd_sections_with_images.json"
PAGE_IMAGE_DIR  = BASE_DIR / "page_images"
CACHE_DIR       = BASE_DIR / "mmrag_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
PAGE_RECORDS_JSON = CACHE_DIR / "page_records.json"
SECTION_EMB_NPY   = CACHE_DIR / "section_embeddings.npy"
PAGE_EMB_NPY      = CACHE_DIR / "page_embeddings.npy"

# ----- Models -----
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
VLM_MODEL_NAME   = "Qwen/Qwen2.5-VL-3B-Instruct"
VLM_DTYPE        = torch.bfloat16 if torch.cuda.is_available() else torch.float32

# ----- Page rendering (only used if page_images/ is missing) -----
RENDER_DPI = 200

# ----- Retrieval -----
TOP_K_TEXT         = 8
TOP_K_PAGE         = 10
FINAL_TEXT_RESULTS = 4
FINAL_PAGE_RESULTS = 4

# ----- Generation -----
MAX_SECTION_TEXT_CHARS = 1800
MAX_PAGE_TEXT_CHARS    = 1200
MAX_NEW_TOKENS         = 320

print("PDF        :", PDF_PATH,        "exists?", PDF_PATH.exists())
print("sections   :", SECTIONS_JSON,   "exists?", SECTIONS_JSON.exists())
print("page imgs  :", PAGE_IMAGE_DIR,  "exists?", PAGE_IMAGE_DIR.exists())
print("cache dir  :", CACHE_DIR)
print("VLM dtype  :", VLM_DTYPE)

## 2. Load section data

Same JSON as in Colab. We normalise the `image_path` so it points at the HPRC `page_images/` folder regardless of what was in the JSON.

In [ ]:
import json

with open(SECTIONS_JSON, "r", encoding="utf-8") as f:
    raw_sections = json.load(f)

sections = []
for sec in raw_sections:
    page_num = int(sec["page_num"])
    expected = PAGE_IMAGE_DIR / f"page_{page_num:04d}.png"
    image_path = sec.get("image_path", "")
    if not image_path or not Path(image_path).exists():
        image_path = str(expected)
    sections.append({
        "section":    str(sec.get("section", "")).strip(),
        "title":      str(sec.get("title", "")).strip(),
        "page_num":   page_num,
        "text":       str(sec.get("text", "")).strip(),
        "image_path": image_path,
    })

print("loaded sections:", len(sections))
print("example keys   :", list(sections[0].keys()) if sections else "(empty)")

## 3. Render page PNGs (only if missing)

On Colab the user produced these elsewhere and uploaded them to Drive.
Here we just render them with PyMuPDF when needed — takes a few minutes for ~1000 pages at 200 DPI on a single CPU core.

In [ ]:
import fitz
from tqdm.auto import tqdm

PAGE_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
doc = fitz.open(str(PDF_PATH))
n_pages = doc.page_count
print(f"PDF pages: {n_pages}")

zoom = RENDER_DPI / 72.0
mat = fitz.Matrix(zoom, zoom)
missing = [i for i in range(n_pages)
           if not (PAGE_IMAGE_DIR / f"page_{i+1:04d}.png").exists()]
if missing:
    print(f"rendering {len(missing)} missing page PNGs at {RENDER_DPI} DPI ...")
    for i in tqdm(missing):
        out = PAGE_IMAGE_DIR / f"page_{i+1:04d}.png"
        pix = doc.load_page(i).get_pixmap(matrix=mat, alpha=False)
        pix.save(str(out))
else:
    print("all page PNGs already present, skipping render.")

## 4. Build page records (text + figure/table captions + sign codes)

Identical to the Colab logic, but cached to disk so re-runs are instant.

In [ ]:
import re

def normalize_spaces(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()

def extract_page_records_from_pdf(pdf_path, page_image_dir):
    doc = fitz.open(str(pdf_path))
    figure_re   = re.compile(r"(Figure\s+[A-Za-z0-9.\-]+[^.\n]*)", re.IGNORECASE)
    table_re    = re.compile(r"(Table\s+[A-Za-z0-9.\-]+[^.\n]*)",  re.IGNORECASE)
    sign_code_re = re.compile(r"\b([A-Z]{1,3}\d{1,2}[A-Za-z0-9\-]*)\b")
    out = []
    for i, page in enumerate(doc):
        page_num = i + 1
        text = page.get_text("text") or ""
        figs = [normalize_spaces(x) for x in figure_re.findall(text)[:3]]
        tbls = [normalize_spaces(x) for x in table_re.findall(text)[:3]]
        signs = sorted(set(sign_code_re.findall(text)))[:50]
        out.append({
            "page_num":        page_num,
            "image_path":      str(page_image_dir / f"page_{page_num:04d}.png"),
            "page_text":       text,
            "page_text_norm":  normalize_spaces(text),
            "figure_captions": figs,
            "table_titles":    tbls,
            "sign_codes":      signs,
        })
    return out

if PAGE_RECORDS_JSON.exists():
    with open(PAGE_RECORDS_JSON, "r", encoding="utf-8") as f:
        page_records = json.load(f)
    print("loaded cached page_records:", len(page_records))
else:
    page_records = extract_page_records_from_pdf(PDF_PATH, PAGE_IMAGE_DIR)
    with open(PAGE_RECORDS_JSON, "w", encoding="utf-8") as f:
        json.dump(page_records, f, ensure_ascii=False)
    print("built and cached page_records:", len(page_records))

## 5. Embeddings — sections + pages

Sentence-transformers MiniLM, cosine via normalized dot product. Cached as `.npy` so subsequent kernel restarts are instant.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

def clean_for_match(text: str) -> str:
    text = (text or "").lower().replace("\u2013", "-").replace("\u2014", "-")
    return re.sub(r"\s+", " ", text).strip()

section_retrieval_texts = [
    f"Section {s['section']}\nTitle: {s['title']}\nPage: {s['page_num']}\n{s['text']}"
    for s in sections
]
page_retrieval_texts = [
    f"Page {p['page_num']}\nFigure captions: {' | '.join(p['figure_captions'])}\n"
    f"Table titles: {' | '.join(p['table_titles'])}\n"
    f"Sign codes: {' | '.join(p['sign_codes'][:20])}\n"
    f"{p['page_text'][:4000]}"
    for p in page_records
]
print("section texts:", len(section_retrieval_texts), " page texts:", len(page_retrieval_texts))

embed_model = SentenceTransformer(EMBED_MODEL_NAME)
if SECTION_EMB_NPY.exists() and PAGE_EMB_NPY.exists():
    section_embeddings = np.load(SECTION_EMB_NPY)
    page_embeddings    = np.load(PAGE_EMB_NPY)
    print("loaded cached embeddings.")
else:
    section_embeddings = embed_model.encode(
        section_retrieval_texts, convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=True,
    ).astype("float32")
    page_embeddings = embed_model.encode(
        page_retrieval_texts, convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=True,
    ).astype("float32")
    np.save(SECTION_EMB_NPY, section_embeddings)
    np.save(PAGE_EMB_NPY, page_embeddings)
    print("computed and saved embeddings.")

print("section_embeddings:", section_embeddings.shape)
print("page_embeddings   :", page_embeddings.shape)

## 6. Query helpers + dual retrieval

In [ ]:
PAGE_QUERY_RE = re.compile(r"\bpage\s+(\d{1,4})\b", re.IGNORECASE)
VISUAL_TERMS = [
    "figure", "fig", "diagram", "table", "chart", "image", "images",
    "show me", "what does this page show", "page ", "sign", "signs",
    "plaque", "plaques", "symbol", "symbols", "label", "labels",
    "caption", "captions", "plate", "plates", "drawing", "drawings",
]

def parse_explicit_page(query):
    m = PAGE_QUERY_RE.search(query or "")
    return int(m.group(1)) if m else None

def is_visual_query(query):
    q = clean_for_match(query)
    return any(term in q for term in VISUAL_TERMS)

def _q_emb(query):
    return embed_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")[0]

def retrieve_text_sections(query, top_k=TOP_K_TEXT):
    qv = _q_emb(query)
    scores = section_embeddings @ qv
    idxs = np.argsort(scores)[::-1][:top_k]
    q_norm = clean_for_match(query)
    results = []
    for idx in idxs:
        sec = sections[idx]
        score = float(scores[idx])
        if q_norm and q_norm in clean_for_match(sec["title"]):              score += 0.20
        if q_norm and q_norm in clean_for_match(sec["text"][:3000]):        score += 0.12
        results.append({"score": score, **sec})
    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]

def retrieve_pages(query, top_k=TOP_K_PAGE):
    qv = _q_emb(query)
    scores = page_embeddings @ qv
    idxs = np.argsort(scores)[::-1][:top_k]
    q_norm = clean_for_match(query)
    visual = is_visual_query(query)
    results = []
    for idx in idxs:
        p = page_records[idx]
        score = float(scores[idx])
        fig_text   = clean_for_match(" | ".join(p["figure_captions"]))
        table_text = clean_for_match(" | ".join(p["table_titles"]))
        label_text = clean_for_match(" | ".join(p["sign_codes"]))
        if q_norm and q_norm in fig_text:               score += 0.30
        if q_norm and q_norm in table_text:             score += 0.25
        if q_norm and q_norm in label_text:             score += 0.16
        if q_norm and q_norm in p["page_text_norm"]:    score += 0.10
        if visual:
            if p["figure_captions"]: score += 0.08
            if p["table_titles"]:    score += 0.06
            if p["sign_codes"]:      score += 0.04
        results.append({"score": score, **p})
    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]

def page_record_by_num(page_num):
    if 1 <= page_num <= len(page_records):
        return page_records[page_num - 1]
    return None

def fuse_results(query):
    explicit_page = parse_explicit_page(query)
    visual = is_visual_query(query)
    text_results = retrieve_text_sections(query, top_k=TOP_K_TEXT)
    page_results = retrieve_pages(query, top_k=TOP_K_PAGE)
    final_text  = text_results[:FINAL_TEXT_RESULTS]
    final_pages = []
    seen = set()
    if explicit_page is not None:
        p = page_record_by_num(explicit_page)
        if p is not None:
            final_pages.append({"score": 999.0, **p}); seen.add(explicit_page)
            for d in (-1, 1):
                npg = explicit_page + d
                n = page_record_by_num(npg)
                if n is not None and npg not in seen:
                    final_pages.append({"score": 998.0 - abs(d), **n}); seen.add(npg)
    if visual:
        for p in page_results:
            if p["page_num"] not in seen:
                final_pages.append(p); seen.add(p["page_num"])
            if len(final_pages) >= FINAL_PAGE_RESULTS: break
    else:
        for t in final_text:
            pg = t["page_num"]
            if pg not in seen:
                p = page_record_by_num(pg)
                if p is not None:
                    final_pages.append({"score": t["score"], **p}); seen.add(pg)
        for p in page_results:
            if p["page_num"] not in seen:
                final_pages.append(p); seen.add(p["page_num"])
            if len(final_pages) >= FINAL_PAGE_RESULTS: break
    return final_text, final_pages[:FINAL_PAGE_RESULTS]

def split_sentences(text):
    raw = (text or "").replace("\n", " ").strip()
    parts = re.split(r"(?<=[.!?])\s+", raw)
    return [p.strip() for p in parts if len(p.strip()) > 20]

def fallback_answer(query, text_results):
    """Extractive fallback used if the VLM fails or returns nothing useful."""
    if not text_results:
        return "No relevant MUTCD sections found."
    evidence = []
    for r in text_results[:3]:
        for s in split_sentences(r["text"])[:8]:
            evidence.append((r, s))
    if not evidence:
        top = text_results[0]
        return (f"Relevant evidence was retrieved from Section {top['section']}, "
                f"Page {top['page_num']}, but I could not form a clean answer.")
    sent_texts = [x[1] for x in evidence]
    sent_emb = embed_model.encode(sent_texts, convert_to_numpy=True,
                                  normalize_embeddings=True).astype("float32")
    qv = _q_emb(query)
    scores = sent_emb @ qv
    order = np.argsort(scores)[::-1]
    chosen, seen = [], set()
    for idx in order:
        r, s = evidence[idx]
        key = (r["section"], s)
        if key in seen: continue
        seen.add(key); chosen.append((r, s))
        if len(chosen) >= 5: break
    answer = " ".join([x[1] for x in chosen])
    cites, seen_cites = [], set()
    for r, _ in chosen:
        c = f"Section {r['section']}, Page {r['page_num']}"
        if c not in seen_cites:
            seen_cites.add(c); cites.append(c)
    if cites:
        answer += "\n\nCitations:\n" + "\n".join(f"- {c}" for c in cites)
    return answer

print("retrieval helpers ready.")

## 7. Smoke test the retrievers (no VLM yet)

Cheap, GPU-free. Useful for confirming the embeddings + JSON loaded correctly before we burn time on the VLM.

In [ ]:
tests = [
    "What does MUTCD say about pedestrian hybrid beacons?",
    "Warning Signs and Plaques for Grade Crossings",
    "What does page 1010 show?",
    "Explain Figure 8C-1",
]
for q in tests:
    t, p = fuse_results(q)
    print("=" * 90)
    print("QUERY:", q)
    print("TEXT TOP:")
    for r in t[:2]:
        print(f"  Section {r['section']:>8s} | {r['title'][:60]:<60s} | Page {r['page_num']:>4d} | s={r['score']:.3f}")
    print("PAGE TOP:")
    for r in p[:3]:
        figs = (r.get('figure_captions') or [''])[0][:60]
        print(f"  Page {r['page_num']:>4d} | s={r.get('score',0):.3f} | {figs}")

## 8. Load the VLM (Qwen2.5-VL-3B-Instruct)

Two paths depending on what's available in your `transformers` version:
1. Preferred: explicit `Qwen2_5_VLForConditionalGeneration` + `AutoProcessor` + `process_vision_info` (from `qwen-vl-utils`). Most robust on HPRC.
2. Fallback: HF `pipeline("image-text-to-text", ...)`, like the Colab notebook.

First time, this downloads ~8 GB into `$HF_HOME`. Make sure `WebProxy` is loaded for the JupyterLab job, or pre-run the snapshot download from a login node (see `HPRC_SETUP.md` §6).

In [ ]:
from PIL import Image

vlm_model = vlm_processor = vlm_pipe = None
_vlm_mode = None

try:
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
    from qwen_vl_utils import process_vision_info

    vlm_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        VLM_MODEL_NAME,
        torch_dtype=VLM_DTYPE,
        device_map="auto",
    )
    vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_NAME)
    _vlm_mode = "explicit"
    print("VLM loaded via explicit Qwen2_5_VL classes.")
except Exception as e:
    print("explicit Qwen2_5_VL path failed, falling back to pipeline:", repr(e))
    from transformers import pipeline
    vlm_pipe = pipeline(
        "image-text-to-text",
        model=VLM_MODEL_NAME,
        device_map="auto",
        torch_dtype=VLM_DTYPE,
    )
    _vlm_mode = "pipeline"
    print("VLM loaded via pipeline.")

print("mode:", _vlm_mode)

## 9. Build VLM messages and answer

Same prompt structure as the Colab notebook, but the generation call branches on whichever loader worked above.

In [ ]:
def build_vlm_messages(question, text_results, page_results):
    content = []
    for p in page_results[:FINAL_PAGE_RESULTS]:
        img_path = p["image_path"]
        if img_path and Path(img_path).exists():
            if _vlm_mode == "explicit":
                content.append({"type": "image", "image": f"file://{img_path}"})
            else:
                content.append({"type": "image", "image": Image.open(img_path)})

    text_blocks = [
        (f"[Text Source {i}]\nSection: {r['section']}\nTitle: {r['title']}\n"
         f"Page: {r['page_num']}\nText:\n{r['text'][:MAX_SECTION_TEXT_CHARS]}")
        for i, r in enumerate(text_results[:FINAL_TEXT_RESULTS], 1)
    ]
    page_blocks = [
        (f"[Page Source {i}]\nPage: {p['page_num']}\n"
         f"Figure captions: {' | '.join(p['figure_captions'])}\n"
         f"Table titles: {' | '.join(p['table_titles'])}\n"
         f"Sign codes: {' | '.join(p['sign_codes'][:20])}")
        for i, p in enumerate(page_results[:FINAL_PAGE_RESULTS], 1)
    ]

    prompt = (
        "You are answering a question using retrieved MUTCD evidence.\n\n"
        "Use both evidence channels equally:\n"
        "- text sources for exact MUTCD wording\n"
        "- page images for figures, diagrams, tables, sign sheets, captions, labels, and page layout\n\n"
        "Rules:\n"
        "1. Use only the provided text and page images.\n"
        "2. If the question is about a specific page or figure, prioritize describing that page.\n"
        "3. If the question is visual, explicitly describe what the relevant page(s) show.\n"
        "4. If the question is textual, use the section wording precisely.\n"
        "5. Write a clear answer in 3 to 7 sentences.\n"
        "6. End with citations.\n"
        "7. Do not invent facts.\n\n"
        f"Question:\n{question}\n\n"
        f"Retrieved text evidence:\n" + "\n".join(text_blocks) + "\n\n"
        f"Retrieved page evidence:\n" + "\n".join(page_blocks) + "\n\n"
        "Output format:\n\nAnswer:\n<clear explanation>\n\nCitations:\n- Section X, Page Y\n- Page Z"
    )
    content.append({"type": "text", "text": prompt})
    return [{"role": "user", "content": content}]

def ask_vlm(question, text_results, page_results):
    messages = build_vlm_messages(question, text_results, page_results)
    if _vlm_mode == "explicit":
        from qwen_vl_utils import process_vision_info
        text = vlm_processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = vlm_processor(
            text=[text], images=image_inputs, videos=video_inputs,
            padding=True, return_tensors="pt",
        ).to(vlm_model.device)
        with torch.inference_mode():
            gen = vlm_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, gen)]
        out_text = vlm_processor.batch_decode(
            trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]
        return out_text
    else:
        output = vlm_pipe(
            text=messages, max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False, return_full_text=False,
        )
        generated = output[0]["generated_text"]
        if isinstance(generated, list):
            return generated[-1]["content"]
        return generated

def answer_question(question):
    question = (question or "").strip()
    if not question:
        return "Please enter a question.", [], ""
    text_results, page_results = fuse_results(question)
    try:
        answer = ask_vlm(question, text_results, page_results)
        if not answer or len(answer.strip()) < 40:
            answer = fallback_answer(question, text_results)
    except Exception as e:
        print("[ask_vlm error]", repr(e))
        answer = fallback_answer(question, text_results)

    gallery_items = []
    for p in page_results:
        ip = p["image_path"]
        if ip and Path(ip).exists():
            cap = (f"Page {p['page_num']} | Figures: {' | '.join(p['figure_captions'])} "
                   f"| Tables: {' | '.join(p['table_titles'])}")
            gallery_items.append((ip, cap))
    ref_lines = []
    for r in text_results:
        ref_lines.append(f"Section {r['section']} \u2014 {r['title']} (Page {r['page_num']})")
    for p in page_results:
        ref_lines.append(f"Page {p['page_num']} \u2014 visual evidence")
    ref_text = "\n".join(dict.fromkeys(ref_lines))
    return answer, gallery_items, ref_text

print("VLM answerer ready.")

## 10. Quick end-to-end sanity check

In [ ]:
answer, gallery, refs = answer_question("What does MUTCD say about pedestrian hybrid beacons?")
print(answer)
print("\n--- references ---")
print(refs)
print("\nGallery items returned:", len(gallery))

## 11. Gradio UI — accessed via OnDemand proxy *or* gradio.live

On HPRC the cleanest way to view this is through the **OnDemand reverse proxy**: any port on your compute node is reachable at

```
https://portal-<cluster>.hprc.tamu.edu/rnode/<hostname>/<port>/
```

The cell below prints that URL and *also* tries `share=True`. The shareable `gradio.live` link only works if `WebProxy` was loaded for the JupyterLab job. If you don't see it, just use the OnDemand URL above — it's faster and stays inside TAMU's network.

In [ ]:
import socket, gradio as gr

# IMPORTANT on HPRC: WebProxy exports http_proxy/https_proxy for outbound
# internet, but Gradio's internal startup handshake hits localhost. If we
# don't tell the system to bypass the proxy for localhost, that handshake
# is routed through the proxy and fails with 503. Set NO_PROXY before
# anything touches the network.
for _k in ("no_proxy", "NO_PROXY"):
    cur = os.environ.get(_k, "")
    extra = "localhost,127.0.0.1,0.0.0.0,::1"
    os.environ[_k] = (cur + "," + extra) if cur else extra

def on_send(message, history):
    ans, gallery_items, refs = answer_question(message)
    final_text = ans + (f"\n\nTop references:\n{refs}" if refs else "")
    history = history + [(message, final_text)]
    return history, "", gallery_items, refs, history

def on_clear():
    return [], "", [], "", []

with gr.Blocks(title="MUTCD Multimodal RAG (HPRC)", theme=gr.themes.Soft()) as demo:
    gr.Markdown("## MUTCD Multimodal RAG \u2014 HPRC")
    gr.Markdown("Dual retrieval: text sections + full pages, fused, then answered by Qwen2.5-VL.")
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=520, label="Chat")
            message = gr.Textbox(label="Question",
                                  placeholder="Example: What does page 1010 show?")
            with gr.Row():
                send_btn  = gr.Button("Send", variant="primary")
                clear_btn = gr.Button("Clear")
        with gr.Column(scale=2):
            gallery = gr.Gallery(label="Relevant MUTCD pages", height=580, preview=True)
            refs    = gr.Textbox(label="References", lines=10)
    state = gr.State([])
    send_btn.click(on_send, [message, state], [chatbot, message, gallery, refs, state])
    message.submit(on_send,  [message, state], [chatbot, message, gallery, refs, state])
    clear_btn.click(on_clear, [],              [chatbot, message, gallery, refs, state])

# Pick a free port and surface BOTH access URLs.
def _free_port():
    s = socket.socket(); s.bind(("", 0)); port = s.getsockname()[1]; s.close(); return port
PORT = int(os.environ.get("MRAG_PORT", _free_port()))
host = socket.gethostname()
for cluster_hint in ("grace", "faster", "launch", "aces"):
    if cluster_hint in host.lower():
        cluster = cluster_hint; break
else:
    cluster = os.environ.get("CLUSTER", "<cluster>")
ondemand_url = f"https://portal-{cluster}.hprc.tamu.edu/rnode/{host}/{PORT}/"
print("OnDemand proxy URL (recommended):", ondemand_url)
print("(Click that ^ inside your OnDemand session; it's private to your TAMU login.)\n")

demo.queue().launch(
    server_name="0.0.0.0",
    server_port=PORT,
    share=True,           # also tries to open a gradio.live tunnel; requires WebProxy.
    inbrowser=False,
    show_error=True,
    root_path=f"/rnode/{host}/{PORT}",
)

## 12. Shutting down

When you're done:
1. **File → Shut Down All Kernels** to free the GPU.
2. From the OnDemand dashboard, click **Delete** on the JupyterLab session so you stop accumulating walltime.
3. Your embeddings + page images stay under `$SCRATCH/MRAG/mmrag_cache` and reload instantly next session.

If you want to keep them past scratch's purge window, copy them to project storage or pull them back to your laptop with `rsync` from the data transfer node (see `HPRC_SETUP.md` §12).